In [ ]:
import pandas as pd 
import json 

## Please Download CSV file before run the notebook
https://drive.google.com/file/d/1AIXZFfy-GQVlNHbofxqYd0MhrmFhJ2Iv/view?usp=drive_link

In [ ]:
inventory_Dataset = pd.read_csv("smartstock_inventory_en.csv")

In [ ]:
inventory_Dataset

In [ ]:
Category_list = inventory_Dataset["Category"].drop_duplicates().to_markdown(index = False )

In [ ]:
def Check_QTY(Category) : 
    df = inventory_Dataset.copy()
    filtered_df = df[df["Category"] == Category][["Product Name", "Unit" , "Available Qty" ]]
    context = filtered_df.to_markdown(index=False)
    return context 

In [ ]:
print ( Check_QTY ( "Personal Care"))

In [ ]:
def get_product_info (product_name) :
    df = inventory_Dataset.copy()
    filtered_df = df[df["Product Name"].str.contains(product_name , case = False)]
    return filtered_df.to_markdown(index = False )


In [ ]:
print (get_product_info("Laundry Powder 3kg"))

In [ ]:
def low_alert (threshold = 20 ) : 
    df = inventory_Dataset.copy()
    filtered_df = df[df["Available Qty"] < threshold ]
    result_df = filtered_df[["Product ID" , "Product Name", "Expiry Date"  , "Storage Location"]]
    return result_df.to_markdown (index = False )

In [ ]:
print (low_alert(60))

In [ ]:
check_qty_function = { 
    "name" : "Check_QTY", 
    "description" : "Check quantity of goods in inventory by category " , 
    "parameters" : { 
        "type" : "object" ,
        "properties" : { 

            "Category" : { 
                "type": "string" ,
                "description" : "name of category user want to check thier quantity"
            }
        }
        , "required" : ["Category"]
    }
}

In [ ]:
get_info_function = { 
    "name" : "get_product_info" , 
    "description" :"get info about product  using product name ",
    "parameters" : { 
        "type" : "object" ,
        "properties" : { 
            "product_name" : { 
                "type" : "string",
                "description" : "name of product user want to check thier info"
            }
        },
    "required" : ["product_name"]
    }
}

In [ ]:
low_alert_function = { 
    "name" : "low_alert" , 
    "description" :"Alerts and retrieves a list of products that are running low in stock below a certain threshold",
    "parameters" : { 
        "type" : "object" ,
        "properties" : { 
            "threshold" : { 
                "type" : "integer",
                "description" : "The safety limit quantity. Defaults to 10 if not specified"
            }
        }
    }
}

In [ ]:
tools = [{"type" : "function" , "function" :check_qty_function }, {"type" : "function" , "function" : get_info_function}, {"type" : "function" , "function" :low_alert_function }]
tools

In [ ]:
def handel_tool_call ( tool_call ) :
    arg = json.loads (tool_call.function.arguments)
    
    if tool_call.function.name == "Check_QTY" : 
        Category_para= arg.get("Category")
        final_output = Check_QTY (Category_para)

    elif tool_call.function.name == "get_product_info" : 
        info_product = arg.get("product_name")
        final_output = get_product_info (info_product )

    elif tool_call.function.name == "low_alert" :
        low_alert_para = arg.get("threshold", 10)
        final_output = low_alert (low_alert_para)
    else :
        final_output = f"Error: Tool name '{tool_call.function.name}' did not match any local functions."
    response = { 
        "role" : "tool", 
        "content" : str  (final_output) , 
        "tool_call_id" : tool_call.id
    }

    return response 

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI 
import gradio as gr 

In [ ]:
load_dotenv (override = True )

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

Model = "gpt-4.1-mini"

openai = OpenAI  ()
    

In [ ]:
system_prompt = f"""
You are SmartStock Assistant,
a professional AI assistant specialized in warehouse and inventory management. You help warehouse managers,
stock controllers, and operations staff monitor inventory, 
answer questions about stock levels,
and execute inventory operations through available tools.

Available Tools
Check_QTY use if user want to check about specicifc category in inventory_Dataset

Behavior & Tone
Respond only in English, regardless of the language the user writes in. 
If the user writes in another language, understand their request but reply in English.
Be concise, accurate, and professional — this is an operational tool, not a casual chat assistant. 
Favor short, clear sentences and structured data (tables) when presenting multiple records.
Use a confident, helpful tone. State numbers and statuses plainly (e.g., "Greek Yogurt 500g is at 40 units, 
below its reorder level of 50 — recommend restocking.").
When presenting inventory data with more than 3-4 fields per item, use a table for clarity.
 For single-item lookups or simple confirmations, respond in plain sentences.
Do not add emojis, exclamation marks, or casual filler. 
Keep responses business-appropriate.

the Categorys user have in thier inventory is 
{Category_list}

"""

In [ ]:
def chat (message , history) : 
    history = [{"role" :h["role"] , "content" : h["content"]} for h in history ]
    messages = [ {"role" : "system" , "content" : system_prompt  } ] + history +[{"role" : "user" , "content" : message }]
    response = openai.chat.completions.create (model = Model , messages = messages , tools = tools )

    if response.choices[0].finish_reason == "tool_calls" : 
        message = response.choices[0].message 
        messages.append(message)
        for tool_call in message.tool_calls : 
            response = handel_tool_call(tool_call)
            messages.append(response)
        response = openai.chat.completions.create (model = Model , messages = messages)
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface (fn = chat , type = "messages").launch(inbrowser=True )